# Day 1: IPL Dataset Audit

## Project
**IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning**

### Day 1 objective
Audit the supplied IPL datasets **after extraction** before building the ingestion pipeline.

This notebook checks:

- Extracted dataset structure
- File counts and sizes
- Cricsheet IPL JSON coverage
- Seasons, dates, players, and teams
- 2026 processed CSV coverage
- Fixture coverage
- Ball-by-ball schema
- Duplicate records
- Batting and bowling scorecards
- Target feasibility
- Last-5-match sequence feasibility
- Data leakage risks
- Day 1 decisions

**Important:** Raw extracted data is read only. This notebook does not clean, delete, overwrite, or modify source files.


## Expected extracted structure

Extract the two ZIP archives first.

Recommended structure:

```text
ipl-player-performance/
└── data/
    └── raw/
        ├── ipl_male_json/
        │   ├── README.txt
        │   ├── 1082591.json
        │   ├── ...
        │   └── 1535465.json
        │
        └── 2026/
            ├── fixtures.csv
            └── matches/
                ├── match_01_.../
                │   ├── ball_by_ball.csv
                │   ├── batting_scorecard.csv
                │   ├── bowling_scorecard.csv
                │   └── ...
                └── match_74_.../
```

The notebook uses these **extracted folders**, not the ZIP files.


In [61]:
from pathlib import Path
import json
import re
import os
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_ROOT = Path.cwd()

# Support launching Jupyter from either the project root or notebooks/.
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CRICSHEET_DIR = RAW_DIR / "ipl_male_json"
DATA_2026_DIR = RAW_DIR / "2026"

print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)
print("Cricsheet directory:", CRICSHEET_DIR)
print("2026 directory:", DATA_2026_DIR)

assert RAW_DIR.exists(), f"Raw directory not found: {RAW_DIR}"
assert CRICSHEET_DIR.exists(), (
    f"Cricsheet extracted directory not found: {CRICSHEET_DIR}"
)
assert DATA_2026_DIR.exists(), (
    f"2026 extracted directory not found: {DATA_2026_DIR}"
)

print("\nExtracted source directories: OK")


Project root: d:\Git\Personal\ipl-player-performance
Raw directory: d:\Git\Personal\ipl-player-performance\data\raw
Cricsheet directory: d:\Git\Personal\ipl-player-performance\data\raw\ipl_male_json
2026 directory: d:\Git\Personal\ipl-player-performance\data\raw\2026

Extracted source directories: OK


In [62]:
# Audit extracted file counts and sizes.

def folder_size_bytes(folder: Path) -> int:
    return sum(
        p.stat().st_size
        for p in folder.rglob("*")
        if p.is_file()
    )

cricsheet_files = [p for p in CRICSHEET_DIR.rglob("*") if p.is_file()]
processed_files = [p for p in DATA_2026_DIR.rglob("*") if p.is_file()]

cricsheet_size_mb = folder_size_bytes(CRICSHEET_DIR) / (1024 ** 2)
processed_size_mb = folder_size_bytes(DATA_2026_DIR) / (1024 ** 2)
combined_size_mb = cricsheet_size_mb + processed_size_mb

print("Cricsheet extracted files:", len(cricsheet_files))
print(f"Cricsheet extracted size: {cricsheet_size_mb:.2f} MB")
print("2026 extracted files:", len(processed_files))
print(f"2026 extracted size: {processed_size_mb:.2f} MB")
print(f"Combined extracted size: {combined_size_mb:.2f} MB")

assert combined_size_mb < 500, "Combined extracted dataset exceeds 500 MB."
print("500 MB requirement: PASS")


Cricsheet extracted files: 1244
Cricsheet extracted size: 100.70 MB
2026 extracted files: 1083
2026 extracted size: 2.54 MB
Combined extracted size: 103.24 MB
500 MB requirement: PASS


## 1. Cricsheet IPL JSON audit

In [63]:
json_files = sorted(CRICSHEET_DIR.glob("*.json"))

print("JSON match files:", len(json_files))

if not json_files:
    raise FileNotFoundError(
        f"No JSON files found directly under {CRICSHEET_DIR}"
    )

# Inspect the first JSON file.
with json_files[0].open("r", encoding="utf-8") as f:
    sample_match = json.load(f)

print("Sample JSON file:", json_files[0].name)
print("Top-level keys:", list(sample_match.keys()))

print("\nSample info keys:")
print(list(sample_match.get("info", {}).keys()))

print("\nSample innings count:", len(sample_match.get("innings", [])))


JSON match files: 1243
Sample JSON file: 1082591.json
Top-level keys: ['meta', 'info', 'innings']

Sample info keys:
['balls_per_over', 'city', 'dates', 'event', 'gender', 'match_type', 'officials', 'outcome', 'overs', 'player_of_match', 'players', 'registry', 'season', 'team_type', 'teams', 'toss', 'venue']

Sample innings count: 2


In [64]:
# Full Cricsheet metadata audit.

seasons = []
match_dates = []
player_names = set()
teams = set()
match_types = set()
registry_person_ids = set()

for path in json_files:
    with path.open("r", encoding="utf-8") as f:
        match = json.load(f)

    info = match.get("info", {})

    season = info.get("season")
    if season is not None:
        seasons.append(season)

    dates = info.get("dates", [])
    for d in dates:
        match_dates.append(str(d))

    match_type = info.get("type")
    if match_type:
        match_types.add(str(match_type))

    for team in info.get("teams", []):
        teams.add(str(team))

    for player_list in info.get("players", {}).values():
        for player in player_list:
            player_names.add(str(player))

    registry = info.get("registry", {})
    people = registry.get("people", {})
    for person_id in people.values():
        registry_person_ids.add(str(person_id))

print("JSON match files:", len(json_files))
print("Unique player names:", len(player_names))
print("Unique teams:", len(teams))
print("Registry person IDs:", len(registry_person_ids))
print("Match types:", sorted(match_types))

if match_dates:
    print("Date range:", min(match_dates), "to", max(match_dates))


JSON match files: 1243
Unique player names: 811
Unique teams: 19
Registry person IDs: 964
Match types: []
Date range: 2008-04-18 to 2026-05-31


In [65]:
# Season counts.
# Normalize mixed Python types before sorting.

season_values = [
    str(season).strip()
    for season in seasons
    if season is not None and str(season).strip()
]

season_counts = pd.Series(
    season_values,
    dtype="string"
).value_counts()

season_counts_df = (
    season_counts
    .rename_axis("Season")
    .reset_index(name="Matches")
)

season_counts_df["Season_Number"] = pd.to_numeric(
    season_counts_df["Season"],
    errors="coerce"
)

season_counts_df = (
    season_counts_df
    .sort_values(["Season_Number", "Season"], na_position="last")
    .drop(columns="Season_Number")
    .reset_index(drop=True)
)

display(season_counts_df)


,Season,Matches
0,2009,57
1,2011,73
2,2012,74
3,2013,76
4,2014,60
5,2015,59
6,2016,60
7,2017,59
8,2018,60
9,2019,60


## 2. 2026 processed dataset audit

In [66]:
# Locate fixtures.csv in the extracted 2026 directory.

fixture_candidates = list(DATA_2026_DIR.rglob("fixtures.csv"))

print("fixtures.csv files found:", len(fixture_candidates))

if not fixture_candidates:
    raise FileNotFoundError(
        f"fixtures.csv not found under {DATA_2026_DIR}"
    )

fixtures_path = fixture_candidates[0]
fixtures = pd.read_csv(fixtures_path)

print("Fixture file:", fixtures_path)
print("Fixture shape:", fixtures.shape)
print("Fixture columns:", fixtures.columns.tolist())

print("\nFixture missing values:")
display(fixtures.isna().sum().to_frame("Missing Values"))

print("Duplicate fixture rows:", fixtures.duplicated().sum())

if "match_number" in fixtures.columns:
    print(
        "Duplicate match numbers:",
        fixtures["match_number"].duplicated().sum()
    )

date_columns = [
    c for c in fixtures.columns
    if "date" in c.lower()
]

if date_columns:
    fixture_date_col = date_columns[0]
    fixtures[fixture_date_col] = pd.to_datetime(
        fixtures[fixture_date_col],
        errors="coerce"
    )
    print(
        "Fixture date range:",
        fixtures[fixture_date_col].min(),
        "to",
        fixtures[fixture_date_col].max()
    )

display(fixtures.head())


fixtures.csv files found: 1
Fixture file: d:\Git\Personal\ipl-player-performance\data\raw\2026\fixtures.csv
Fixture shape: (70, 5)
Fixture columns: ['match_number', 'date', 'venue', 'team_1', 'team_2']

Fixture missing values:


,Missing Values
match_number,0
date,0
venue,0
team_1,0
team_2,0


Duplicate fixture rows: 0
Duplicate match numbers: 0
Fixture date range: 2026-03-28 00:00:00 to 2026-05-24 00:00:00


,match_number,date,venue,team_1,team_2
0,1,2026-03-28,"M Chinnaswamy Stadium, Bengaluru",Royal Challengers Bengaluru,Sunrisers Hyderabad
1,2,2026-03-29,"Wankhede Stadium, Mumbai",Mumbai Indians,Kolkata Knight Riders
2,3,2026-03-30,"ACA Stadium, Guwahati",Rajasthan Royals,Chennai Super Kings
3,4,2026-03-31,"New International Cricket Stadium, New Chandigarh",Punjab Kings,Gujarat Titans
4,5,2026-04-01,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow Super Giants,Delhi Capitals


In [67]:
# Count match directories.

match_dirs = sorted(
    p for p in (DATA_2026_DIR / "matches").iterdir()
    if p.is_dir()
)

print("2026 match directories:", len(match_dirs))
print("First 5:")
for p in match_dirs[:5]:
    print(" ", p.name)

print("\nLast 10:")
for p in match_dirs[-10:]:
    print(" ", p.name)

print(
    "\nFixture rows vs match directories:",
    len(fixtures),
    "vs",
    len(match_dirs)
)

if len(fixtures) != len(match_dirs):
    print(
        "FINDING: fixtures.csv does not represent every extracted "
        "2026 match directory. Day 2 must reconcile match metadata."
    )


2026 match directories: 74
First 5:
  match_01_Sunrisers_Hyderabad_vs_Royal_Challengers_Bengaluru
  match_02_Kolkata_Knight_Riders_vs_Mumbai_Indians
  match_03_Chennai_Super_Kings_vs_Rajasthan_Royals
  match_04_Gujarat_Titans_vs_Punjab_Kings
  match_05_Lucknow_Super_Giants_vs_Delhi_Capitals

Last 10:
  match_65_Mumbai_Indians_vs_Kolkata_Knight_Riders
  match_66_Gujarat_Titans_vs_Chennai_Super_Kings
  match_67_Sunrisers_Hyderabad_vs_Royal_Challengers_Bengaluru
  match_68_Lucknow_Super_Giants_vs_Punjab_Kings
  match_69_Rajasthan_Royals_vs_Mumbai_Indians
  match_70_Delhi_Capitals_vs_Kolkata_Knight_Riders
  match_71_Royal_Challengers_Bengaluru_vs_Gujarat_Titans
  match_72_Rajasthan_Royals_vs_Sunrisers_Hyderabad
  match_73_Rajasthan_Royals_vs_Gujarat_Titans
  match_74_Gujarat_Titans_vs_Royal_Challengers_Bengaluru

Fixture rows vs match directories: 70 vs 74
FINDING: fixtures.csv does not represent every extracted 2026 match directory. Day 2 must reconcile match metadata.


In [68]:
# Audit the 2026 file types across match directories.

file_type_counts = {}

for match_dir in match_dirs:
    for path in match_dir.iterdir():
        if path.is_file():
            file_type_counts[path.name] = (
                file_type_counts.get(path.name, 0) + 1
            )

file_type_df = (
    pd.Series(file_type_counts)
    .rename_axis("File")
    .reset_index(name="Match Files")
    .sort_values("File")
    .reset_index(drop=True)
)

display(file_type_df)


,File,Match Files
0,ball_by_ball.csv,74
1,batting_scorecard.csv,74
2,bowling_scorecard.csv,74
3,fall_of_wickets.csv,74
4,partnerships.csv,74
5,phase_summary.csv,74
6,reviews.csv,62
7,substitutions.csv,73
8,super_over.csv,1


## 3. Ball-by-ball schema and duplicate audit

In [69]:
ball_paths = sorted(
    DATA_2026_DIR.rglob("ball_by_ball.csv")
)

print("Ball-by-ball files:", len(ball_paths))

ball_frames = []

for path in ball_paths:
    df = pd.read_csv(path)
    df["_source_file"] = str(path.relative_to(DATA_2026_DIR))
    ball_frames.append(df)

ball_by_ball = pd.concat(
    ball_frames,
    ignore_index=True
)

print("Ball-by-ball rows:", len(ball_by_ball))
print("Columns:")
print(ball_by_ball.columns.tolist())

print("\nMissing values:")
display(
    ball_by_ball.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

print("Exact duplicate rows:", ball_by_ball.duplicated().sum())

duplicate_by_file = (
    ball_by_ball
    .groupby("_source_file", sort=False)
    .apply(lambda x: x.duplicated().sum())
    .rename("Duplicate Rows")
    .reset_index()
)

duplicate_by_file = duplicate_by_file[
    duplicate_by_file["Duplicate Rows"] > 0
]

print("Files containing duplicate rows:", len(duplicate_by_file))
display(duplicate_by_file.head(20))


Ball-by-ball files: 74
Ball-by-ball rows: 17523
Columns:
['innings', 'team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batter_runs', 'extra_runs', 'total_runs', 'is_boundary', 'non_boundary_run', 'extra_type', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'is_wicket', 'wicket_kind', 'player_out', 'fielder', 'phase', '_source_file']

Missing values:


,Missing Values
fielder,16816
wicket_kind,16643
player_out,16643
extra_type,16429
innings,0
team,0
over,0
ball,0
batter_runs,0
non_striker,0


Exact duplicate rows: 51
Files containing duplicate rows: 36


,_source_file,Duplicate Rows
4,matches\match_05_Lucknow_Super_Giants_vs_Delhi...,1
5,matches\match_06_Sunrisers_Hyderabad_vs_Kolkat...,1
6,matches\match_07_Chennai_Super_Kings_vs_Punjab...,1
13,matches\match_14_Gujarat_Titans_vs_Delhi_Capit...,1
16,matches\match_17_Sunrisers_Hyderabad_vs_Punjab...,3
19,matches\match_20_Royal_Challengers_Bengaluru_v...,1
20,matches\match_21_Sunrisers_Hyderabad_vs_Rajast...,1
21,matches\match_22_Chennai_Super_Kings_vs_Kolkat...,1
26,matches\match_27_Sunrisers_Hyderabad_vs_Chenna...,1
27,matches\match_28_Rajasthan_Royals_vs_Kolkata_K...,2


### Duplicate-data decision

Exact duplicate ball-by-ball rows are **not deleted on Day 1**.

They must be investigated during Day 2 to determine whether they are true source duplicates or represent a valid source structure.

This prevents silent data loss before the ingestion rules are defined.


## 4. Batting scorecard audit

In [70]:
batting_paths = sorted(
    DATA_2026_DIR.rglob("batting_scorecard.csv")
)

batting_frames = [
    pd.read_csv(path)
    for path in batting_paths
]

batting = pd.concat(
    batting_frames,
    ignore_index=True
)

print("Batting scorecard files:", len(batting_paths))
print("Rows:", len(batting))
print("Columns:", batting.columns.tolist())
print("Unique batters:", batting["batter"].nunique())

print("\nMissing values:")
display(
    batting.isna().sum().to_frame("Missing Values")
)

print("Duplicate complete rows:", batting.duplicated().sum())

numeric_candidates = [
    c for c in ["runs", "balls", "fours", "sixes"]
    if c in batting.columns
]

if numeric_candidates:
    print("\nBatting totals:")
    display(batting[numeric_candidates].sum().to_frame("Total"))


Batting scorecard files: 74
Rows: 1130
Columns: ['innings', 'team', 'batter', 'runs', 'balls', 'fours', 'sixes', 'strike_rate', 'dismissal', 'batting_position']
Unique batters: 176

Missing values:


,Missing Values
innings,0
team,0
batter,0
runs,0
balls,0
fours,0
sixes,0
strike_rate,0
dismissal,0
batting_position,0


Duplicate complete rows: 0

Batting totals:


,Total
runs,26151
balls,16727
fours,2332
sixes,1426


## 5. Bowling scorecard audit

In [71]:
bowling_paths = sorted(
    DATA_2026_DIR.rglob("bowling_scorecard.csv")
)

bowling_frames = [
    pd.read_csv(path)
    for path in bowling_paths
]

bowling = pd.concat(
    bowling_frames,
    ignore_index=True
)

print("Bowling scorecard files:", len(bowling_paths))
print("Rows:", len(bowling))
print("Columns:", bowling.columns.tolist())
print("Unique bowlers:", bowling["bowler"].nunique())

print("\nMissing values:")
display(
    bowling.isna().sum().to_frame("Missing Values")
)

print("Duplicate complete rows:", bowling.duplicated().sum())

numeric_candidates = [
    c for c in ["runs", "wickets"]
    if c in bowling.columns
]

if numeric_candidates:
    print("\nBowling totals:")
    display(bowling[numeric_candidates].sum().to_frame("Total"))


Bowling scorecard files: 74
Rows: 854
Columns: ['innings', 'team', 'bowler', 'overs', 'maidens', 'runs', 'wickets', 'economy', 'dots', 'wides', 'noballs']
Unique bowlers: 125

Missing values:


,Missing Values
innings,0
team,0
bowler,0
overs,0
maidens,0
runs,0
wickets,0
economy,0
dots,0
wides,0


Duplicate complete rows: 0

Bowling totals:


,Total
runs,27125
wickets,835


## 6. Additional 2026 source audit

In [72]:
additional_files = {
    "partnerships": sorted(DATA_2026_DIR.rglob("partnerships.csv")),
    "phase_summary": sorted(DATA_2026_DIR.rglob("phase_summary.csv")),
    "fall_of_wickets": sorted(DATA_2026_DIR.rglob("fall_of_wickets.csv")),
    "substitutions": sorted(DATA_2026_DIR.rglob("substitutions.csv")),
    "reviews": sorted(DATA_2026_DIR.rglob("reviews.csv")),
    "super_over": sorted(DATA_2026_DIR.rglob("super_over.csv")),
}

summary_rows = []

for name, paths in additional_files.items():
    rows = None
    columns = None

    if paths:
        sample = pd.read_csv(paths[0])
        rows = sum(len(pd.read_csv(p)) for p in paths)
        columns = len(sample.columns)

    summary_rows.append({
        "Source": name,
        "Files": len(paths),
        "Total Rows": rows,
        "Columns in First File": columns,
    })

display(pd.DataFrame(summary_rows))


,Source,Files,Total Rows,Columns in First File
0,partnerships,74,1007,12
1,phase_summary,74,434,9
2,fall_of_wickets,74,880,6
3,substitutions,73,144,7
4,reviews,62,118,10
5,super_over,1,4,13


## 7. Target feasibility

The proposed supervised targets are:

1. `next_match_runs`
2. `next_match_wickets`
3. `next_match_runs_conceded`

For player-match record **N**, the target must come from the player's next chronological eligible match appearance.

The current Day 1 source audit supports these concepts:

- batting scorecards provide player-match runs
- bowling scorecards provide player-match wickets and runs conceded
- Cricsheet provides historical match and delivery data
- chronological player-match sequences can be constructed

Rows without a subsequent eligible player appearance will not have a future target.


In [73]:
# Basic target feasibility checks from the 2026 scorecards.

print("Batting records available for next_match_runs:", len(batting))
print("Bowling records available for next_match_wickets:", len(bowling))
print("Bowling records available for next_match_runs_conceded:", len(bowling))

print("\nTarget construction status:")
print("next_match_runs: FEASIBLE")
print("next_match_wickets: FEASIBLE")
print("next_match_runs_conceded: FEASIBLE")


Batting records available for next_match_runs: 1130
Bowling records available for next_match_wickets: 854
Bowling records available for next_match_runs_conceded: 854

Target construction status:
next_match_runs: FEASIBLE
next_match_wickets: FEASIBLE
next_match_runs_conceded: FEASIBLE


## 8. Last-5-match feasibility

For each prediction row, the Deep Learning sequence will use the player's previous five chronological eligible matches.

Rules:

- The sequence must contain only matches before the prediction match.
- No fabricated performance values will be used for missing history.
- Players with fewer than five eligible previous matches require an explicit minimum-history rule.
- Sequence ordering must be chronological.


## 9. Leakage risks identified on Day 1

The following rules are mandatory for later days:

- The target match performance cannot be used as an input feature.
- Rolling features must be shifted and use prior matches only.
- Season aggregates must use only information available before the prediction match.
- Venue aggregates must use prior observations only.
- Opposition aggregates must use prior observations only.
- Train, validation, and test sets must be chronological.
- Last-5 DL sequences must contain only previous matches.
- Future 2026 playoff information must not leak into earlier prediction rows.


## 10. Day 1 findings

Based on the extracted source files:

- Cricsheet is the primary historical IPL source.
- The supplied 2026 archive is a structured processed/reference source.
- The raw extracted data remains unchanged.
- The 2026 archive must be reconciled because `fixtures.csv` may not represent every match directory.
- Duplicate ball-by-ball records must be investigated before cleaning.
- Player identity must be standardized during ingestion.
- Match metadata must be constructed from authoritative match-level information.
- Historical schema mapping must be based on actual source fields.
- No ML or DL model is trained on Day 1.


## 11. Day 1 decision log

1. Use extracted Cricsheet IPL JSON as the primary historical source.
2. Use extracted 2026 CSV data as a processed/reference source.
3. Preserve the extracted raw source files unchanged.
4. Investigate duplicate ball-by-ball rows during Day 2.
5. Reconcile the 2026 match directories with `fixtures.csv` during Day 2.
6. Standardize player identity during ingestion.
7. Do not invent player IDs or model features.
8. Do not train models on Day 1.
9. Build the production ingestion pipeline only after the source audit is complete.


# Day 1 conclusion

The extracted datasets provide the required foundation for the IPL-only project.

**Day 1 is complete when every cell in this notebook runs successfully from a fresh kernel and the findings are reviewed.**

